## Preparations

### Импорты

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings("ignore")

### Константы

In [ ]:
ROOT = ".."
DATA_PATH = os.path.join(ROOT, "data", "inference.parquet")

TEXT_COL = "text"
TARGET_COL = "target"

BATCH_SIZE = 32
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### Функции

In [ ]:
@torch.inference_mode()
def run_local_inference(
    inference_df: pd.DataFrame, 
    model_path: str, 
    model_name: str, 
    text_col: str = "text",
    batch_size: int = 32, 
    max_length: int = 512, 
    device: str = DEVICE, 
    verbose: bool = True
) -> tuple:
    """Локальный инференс модели"""
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
    model.to(device)
    model.eval()

    id2label = {int(k): v for k, v in model.config.id2label.items()}
    if verbose:
        print(f"{model_name}: загружена на {device} | id2label={id2label}")

    texts = inference_df[text_col].fillna("").astype(str).tolist()
    # модель не знает про max_length больше своего максимума
    max_length = min(max_length, model.config.max_position_embeddings)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.pad_token_id

    pred_ids, pred_probs = [], []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        probs = torch.softmax(model(**enc).logits, dim=-1)
        pred_ids.extend(probs.argmax(dim=-1).cpu().numpy().tolist())
        pred_probs.extend(probs[:, 1].cpu().numpy().tolist())

        if verbose:
            print(f"\r {min(start + batch_size, len(texts))}/{len(texts)}", end="")
    if verbose:
        print()

    out_df = inference_df.copy()
    out_df[f"{model_name}_pred"] = np.asarray(pred_ids, dtype="int8")
    out_df[f"{model_name}_proba_positive"] = pred_probs

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    return out_df, id2label

In [ ]:
def calcul_metrics(
    tn: int, 
    fp: int, 
    fn: int, 
    tp: int,
    model_name: str
) -> tuple: 
    """Расчет метрик"""
    precision_pos = tp / (tp + fp)
    precision_neg = tn / (tn + fn)
    recall_pos = tp / (tp + fn)
    recall_neg = tn / (tn + fp)
    accuracy = (tp + tn) / (tp + fn + tn + fp)

    return precision_pos, precision_neg, recall_pos, recall_neg, accuracy, model_name

### Заугрузка данных

In [ ]:
inference_df = pd.read_parquet(DATA_PATH, columns=["text_ru", "label", "filter"]).rename(columns={"text_ru": "text"})
inference_df = inference_df[inference_df["filter"] != 1].drop(columns=["filter"]).reset_index(drop=True)
inference_df["target"] = inference_df["label"].replace({"injection": 1, "benign": 0}).astype("int8")

print(inference_df.shape)
inference_df.head()

## https://huggingface.co/deepset/deberta-v3-base-injection/

In [ ]:
MODEL_NAME = "deepset-deberta-v3-base-injection"
MODEL_PATH = os.path.join(ROOT, "models", MODEL_NAME)

inference_df, id2label = run_local_inference(
    inference_df,
    model_path=MODEL_PATH,
    model_name=MODEL_NAME,
    text_col=TEXT_COL,
    batch_size=BATCH_SIZE,
)

inference_df.head(3)

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    inference_df[TARGET_COL],
    inference_df[f"{MODEL_NAME}_pred"],
    labels=[0, 1],
).ravel()

print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")

In [ ]:
precision_pos, precision_neg, recall_pos, recall_neg, accuracy, model_name = calcul_metrics(tn, fp, fn, tp, MODEL_NAME)

In [ ]:
metrics_deepset_df = pd.DataFrame({
    "model_name": [model_name],
    "precision_positive": [precision_pos],
    "precision_negative": [precision_neg],
    "recall_positive": [recall_pos],
    "recall_negative": [recall_neg],
    "accuracy": [accuracy]
})

In [ ]:
metrics_deepset_df

## https://huggingface.co/jackhhao/jailbreak-classifier

In [ ]:
MODEL_NAME = "jackhhao-jailbreak-classifier"
MODEL_PATH = os.path.join(ROOT, "models", MODEL_NAME)

inference_df, id2label = run_local_inference(
    inference_df,
    model_path=MODEL_PATH,
    model_name=MODEL_NAME,
    text_col=TEXT_COL,
    batch_size=BATCH_SIZE,
)

inference_df.head()

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    inference_df[TARGET_COL],
    inference_df[f"{MODEL_NAME}_pred"],
    labels=[0, 1],
).ravel()

print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")

In [ ]:
precision_pos, precision_neg, recall_pos, recall_neg, accuracy, model_name = calcul_metrics(tn, fp, fn, tp, MODEL_NAME)

In [ ]:
metrics_jackhhao_df = pd.DataFrame({
    "model_name": [model_name],
    "precision_positive": [precision_pos],
    "precision_negative": [precision_neg],
    "recall_positive": [recall_pos],
    "recall_negative": [recall_neg],
    "accuracy": [accuracy]
})

In [ ]:
metrics_jackhhao_df

## https://huggingface.co/qualifire/prompt-injection-sentinel

In [ ]:
MODEL_NAME = "qualifire-prompt-injection-sentinel"
MODEL_PATH = os.path.join(ROOT, "models", MODEL_NAME)

inference_df, id2label = run_local_inference(
    inference_df,
    model_path=MODEL_PATH,
    model_name=MODEL_NAME,
    text_col=TEXT_COL,
    batch_size=BATCH_SIZE,
)

inference_df.head()

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    inference_df[TARGET_COL],
    inference_df[f"{MODEL_NAME}_pred"],
    labels=[0, 1],
).ravel()

print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")

In [ ]:
precision_pos, precision_neg, recall_pos, recall_neg, accuracy, model_name = calcul_metrics(tn, fp, fn, tp, MODEL_NAME)

In [ ]:
metrics_qualifire_df = pd.DataFrame({
    "model_name": [model_name],
    "precision_positive": [precision_pos],
    "precision_negative": [precision_neg],
    "recall_positive": [recall_pos],
    "recall_negative": [recall_neg],
    "accuracy": [accuracy]
})

In [ ]:
metrics_qualifire_df

### Собственная дообученная модель prompt-injection (ruBert)

In [ ]:
inference_df, id2label = run_local_inference(
    inference_df,
    model_path=FINETUNED_MODEL_PATH,
    model_name=FINETUNED_MODEL_NAME,
    text_col=TEXT_COL,
    batch_size=EVAL_BATCH_SIZE,
)

inference_df.head()

## Метрики

In [ ]:
metrics_df = pd.concat([metrics_deepset_df, metrics_jackhhao_df, metrics_qualifire_df], axis=0, ignore_index=True)

In [ ]:
metrics_df